# Determinants of emissions and industry descriptives (US)

Sustainable-finance dataset (`44_build_sustainability_dataset.py`). Functions in `src/datastream/emissions_analysis.py`;
this notebook configures and displays. Outputs go to `OUTPUT_DIR`.

**Design**
- **Firm-years, not firm-months.** Emissions are annual; the monthly dataset repeats each value up to 12 times.
  Per firm and fiscal year the first month in which the emission value is used is taken. Emissions, sales and
  fundamentals belong to the same fiscal year (fiscal year end from `WC05350`); market cap at the fiscal year end.
- **Financials excluded** (TRBC economic sector "Financials"): sales of banks and insurers contain interest and
  premium income, so intensities are not comparable; stocks without TRBC classification are dropped as well.
- **Fixed effects:** TRBC industry group and fiscal year.
- **Standard errors:** two-way clustered by firm and fiscal year (emissions are very persistent within firms);
  robustness: industry x year and firm only (section 4).
- **Samples:** all firms (reported or estimated emissions, with a dummy for estimates) and reported emissions only.
  Estimates are model-based and partly a function of size and industry, so results driven by them are suspect.
- Dependent variables (log emissions, log intensities) winsorized 1%/99% per fiscal year (`WINSOR_DEPENDENT`):
  firms with almost no sales or emissions produce extreme log values. Descriptive tables use the raw levels.
- Controls (winsorized 1%/99% per fiscal year, except the logs of sales and market cap): `log_sales`, `log_me`,
  `log_bm` (common equity / market cap at fiscal year end), `leverage`, `invest_ta` (capex / assets), `roe`,
  `sales_growth`.
- Intensity = emissions / sales, so in a regression of log intensity on log sales the coefficient equals the
  emission elasticity minus 1.

Regression tables: coefficient with stars (10% / 5% / 1%), t-statistic in parentheses; blank = regressor not
in the specification (CSV files keep the raw numbers). Scope 1 / scope 2 exist only for firms that report them
(Refinitiv estimates only the total), so the scope regressions are essentially reported-emission samples.

**Expectations:** absolute emissions rise with size (log sales, log market cap) and with capital intensity
(investment); intensities depend mainly on the industry (high R2 of the fixed effects alone).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from datastream import emissions_analysis as ea
from datastream import firm_evaluation as fe

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
fe.set_style()

# ── configuration ─────────────────────────────────────────────────────────────────────────────────────
REGION = "US"
FIRM_ROOT = Path(os.environ.get("DS_FIRM_ROOT", "D:/Datastream/Firmcharacteristics_Monthly")) / REGION
NAME = "lag12_0.25"
DATASET = FIRM_ROOT / "Paneldata" / "sustainability" / f"sustainability_{NAME}.parquet"
EMISSIONS_PANEL = FIRM_ROOT / "Paneldata" / "emissions" / f"emissions_{NAME}.parquet"   # market cap at FYE
FIRST_FISCAL_YEAR = 2002
CLUSTER_MAIN = ("DSCD", "fiscal_year")
FE = ("trbc_industry_group", "fiscal_year")
MIN_INDUSTRY_FIRMS = 5           # industry tables
TOP_N = 15
WINSOR = (0.01, 0.99)            # per fiscal year
WINSOR_DEPENDENT = True
OUTPUT_DIR = FIRM_ROOT / "Paneldata" / "analyses" / "emissions_determinants"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Firm-year sample

In [ ]:
ds = pd.read_parquet(DATASET)
em_panel = pd.read_parquet(EMISSIONS_PANEL, columns=["DSCD", "Date", "MarketCAP"])
fy, counts = ea.firm_year_sample(ds, em_panel, winsor=WINSOR, winsor_dependent=WINSOR_DEPENDENT)
fy = fy[fy["fiscal_year"] >= FIRST_FISCAL_YEAR]
wins = counts.pop("winsorized_dependent", None)
display(pd.Series(counts, name="selection").to_frame())
if wins:
    print("Winsorized firm-years per dependent variable:", wins)
rep = fy[fy["co2e_is_estimated"] == 0]
print(f"Firm-years: {len(fy):,} ({fy['DSCD'].nunique():,} firms), reported: {len(rep):,} ({rep['DSCD'].nunique():,} firms); "
      f"fiscal years {fy['fiscal_year'].min()}-{fy['fiscal_year'].max()}; "
      f"{fy['trbc_industry_group'].nunique()} TRBC industry groups")
per_year = pd.DataFrame({"firm_years": fy.groupby("fiscal_year").size(),
                         "reported": rep.groupby("fiscal_year").size()}).fillna(0).astype(int)
display(per_year.T)

In [ ]:
summary_vars = ["log_co2e_total_combined", "log_co2e_scope1", "log_co2e_scope2", "log_co2e_intensity",
                "co2e_intensity", "co2e_is_estimated"] + ea.CONTROLS
stats = pd.concat({"all": fy[summary_vars].describe(percentiles=[0.05, 0.5, 0.95]).T,
                   "reported": rep[summary_vars].describe(percentiles=[0.05, 0.5, 0.95]).T}, axis=1)
stats.to_csv(OUTPUT_DIR / "1_summary_statistics.csv")
display(stats)

In [ ]:
# where the regression sample is lost (full specification of section 2, all firms)
miss = ea.missing_overview(fy, "log_co2e_total_combined", ea.CONTROLS)
miss.to_csv(OUTPUT_DIR / "1_missing_overview.csv")
display(miss)

In [ ]:
corr = fy[["log_co2e_total_combined", "log_co2e_intensity"] + ea.CONTROLS].corr()
corr.to_csv(OUTPUT_DIR / "1_correlations.csv")
display(corr)

## 2. Absolute emissions
Log total CO2e (reported, else estimated). Columns (1)-(4): all firms with a dummy for estimated emissions;
(5): reported emissions only. Coefficients with t-statistics (two-way clustered by firm and fiscal year) below.
`R2 FE only`: R2 of the industry and year fixed effects alone on the same sample.

In [ ]:
y = "log_co2e_total_combined"
specs_abs = {
    "(1) all": (y, ["log_sales", "co2e_is_estimated"]),
    "(2) all": (y, ["log_me", "co2e_is_estimated"]),
    "(3) all": (y, ["log_sales", "log_me", "co2e_is_estimated"]),
    "(4) all": (y, ea.CONTROLS + ["co2e_is_estimated"]),
}
tab_abs = ea.regression_table(fy, specs_abs, fe=FE, cluster=CLUSTER_MAIN)
tab_abs["(5) reported"] = ea.regression_table(rep, {"r": (y, ea.CONTROLS)}, fe=FE, cluster=CLUSTER_MAIN)["r"]
tab_abs.to_csv(OUTPUT_DIR / "2_absolute_total.csv")
display(ea.format_table(tab_abs))

Scope 1 and scope 2 separately (full controls).

In [ ]:
specs_scope = {}
for dep in ["log_co2e_scope1", "log_co2e_scope2"]:
    specs_scope[f"{dep} all"] = (dep, ea.CONTROLS + ["co2e_is_estimated"])
tab_scope = ea.regression_table(fy, specs_scope, fe=FE, cluster=CLUSTER_MAIN)
for dep in ["log_co2e_scope1", "log_co2e_scope2"]:
    tab_scope[f"{dep} reported"] = ea.regression_table(rep, {"r": (dep, ea.CONTROLS)}, fe=FE, cluster=CLUSTER_MAIN)["r"]
tab_scope.to_csv(OUTPUT_DIR / "2_absolute_scopes.csv")
display(ea.format_table(tab_scope))

## 3. Intensities
Log emissions per USD million of sales of the same fiscal year. The coefficient on `log_sales` is the emission
elasticity minus 1 (a negative value means that intensity falls with size).

In [ ]:
specs_int = {}
for dep in ["log_co2e_intensity", "log_co2e_scope1_intensity"]:
    specs_int[f"{dep} (a) all"] = (dep, ["log_me", "co2e_is_estimated"])
    specs_int[f"{dep} (b) all"] = (dep, ea.CONTROLS + ["co2e_is_estimated"])
tab_int = ea.regression_table(fy, specs_int, fe=FE, cluster=CLUSTER_MAIN)
for dep in ["log_co2e_intensity", "log_co2e_scope1_intensity"]:
    tab_int[f"{dep} (b) reported"] = ea.regression_table(rep, {"r": (dep, ea.CONTROLS)}, fe=FE, cluster=CLUSTER_MAIN)["r"]
tab_int.to_csv(OUTPUT_DIR / "3_intensities.csv")
display(ea.format_table(tab_int))

## 4. Robustness: clustering
t-statistics of the full specification (all firms) under different clusterings.

In [ ]:
rob = {}
for dep in ["log_co2e_total_combined", "log_co2e_scope1", "log_co2e_intensity"]:
    for label, cl in {"firm x year": ("DSCD", "fiscal_year"), "industry x year": ("trbc_industry_group", "fiscal_year"),
                      "firm": ("DSCD",), "industry": ("trbc_industry_group",)}.items():
        res, _ = ea.fe_regression(fy, dep, ea.CONTROLS + ["co2e_is_estimated"], fe=FE, cluster=cl)
        rob[(dep, label)] = res.tvalues[ea.CONTROLS + ["co2e_is_estimated"]]
rob = pd.DataFrame(rob)
rob.to_csv(OUTPUT_DIR / "4_clustering_tstats.csv")
display(rob)

## 5. Industries with the highest emissions and intensities
Latest fiscal year with nearly complete coverage (at least 90% of the maximum number of firm-years), TRBC
industry groups with at least `MIN_INDUSTRY_FIRMS` firms. `total_mt`: sum of emissions (Mt CO2e);
`share_of_total`: share of all emissions of the year; `median_kt`: median per firm (kt);
`aggregate_intensity`: sum of emissions / sum of sales (t / USD m).

In [ ]:
n_by_year = fy.groupby("fiscal_year").size()
DESC_YEAR = int(n_by_year[n_by_year >= 0.9 * n_by_year.max()].index.max())
print(f"Descriptive year: fiscal {DESC_YEAR} ({n_by_year[DESC_YEAR]:,} firms)")
cur = fy[fy["fiscal_year"] == DESC_YEAR]
ind_all = ea.industry_table(cur, min_firms=MIN_INDUSTRY_FIRMS)
ind_rep = ea.industry_table(cur[cur["co2e_is_estimated"] == 0], min_firms=MIN_INDUSTRY_FIRMS)
ind_all.to_csv(OUTPUT_DIR / f"5_industries_{DESC_YEAR}_all.csv")
ind_rep.to_csv(OUTPUT_DIR / f"5_industries_{DESC_YEAR}_reported.csv")
print("Highest total emissions (all firms):")
display(ind_all.sort_values("total_mt", ascending=False).head(TOP_N))
print("Highest median intensity (all firms):")
display(ind_all.sort_values("median_intensity", ascending=False).head(TOP_N))
print("Highest median intensity (reported only):")
display(ind_rep.sort_values("median_intensity", ascending=False).head(TOP_N))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
a = ind_all.sort_values("total_mt", ascending=False).head(TOP_N)[::-1]
axes[0].barh(a.index, a["total_mt"], color=fe.CATEGORICAL[0])
axes[0].set_title(f"Total emissions, fiscal {DESC_YEAR} (Mt CO2e)", loc="left")
b = ind_all.sort_values("median_intensity", ascending=False).head(TOP_N)[::-1]
axes[1].barh(b.index, b["median_intensity"], color=fe.CATEGORICAL[1])
axes[1].set_title(f"Median intensity, fiscal {DESC_YEAR} (t / USD m sales)", loc="left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"5_top_industries_{DESC_YEAR}.png", dpi=130, bbox_inches="tight")
plt.show()

Economic sectors, pooled over fiscal years 2016 onwards (broad coverage), and concentration of emissions.

In [ ]:
pooled = fy[fy["fiscal_year"] >= 2016]
sec = ea.industry_table(pooled, level="trbc_economic_sector", min_firms=1)
sec["firm_years"] = pooled.groupby("trbc_economic_sector").size()
sec.to_csv(OUTPUT_DIR / "5_sectors_2016plus.csv")
display(sec.sort_values("total_mt", ascending=False))
e = cur["co2e_total_combined"].sort_values(ascending=False)
conc = pd.Series({f"top {q:.0%} of firms": e.head(int(np.ceil(q * len(e)))).sum() / e.sum() for q in (0.01, 0.05, 0.10, 0.25)},
                 name=f"share of total emissions, fiscal {DESC_YEAR}")
display(conc.to_frame())

Aggregate intensity by economic sector over time (reported emissions only; composition changes over time).

In [ ]:
ts = ea.aggregate_intensity_by_year(fy[fy["fiscal_year"] <= DESC_YEAR], reported_only=True)
ts.to_csv(OUTPUT_DIR / "5_sector_intensity_by_year.csv")
fig, ax = plt.subplots(figsize=(11, 4.5))
for i, c in enumerate(ts.columns):
    ax.plot(ts.index, ts[c], label=c, color=fe.CATEGORICAL[i % len(fe.CATEGORICAL)], lw=1.6)
ax.set_yscale("log")
ax.set_ylabel("t CO2e / USD m sales (log scale)")
ax.set_title("Aggregate intensity by TRBC economic sector (reported emissions)", loc="left")
ax.legend(ncol=2, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "5_sector_intensity_by_year.png", dpi=130, bbox_inches="tight")
plt.show()